In [ ]:
import pandas as pd
import mysql.connector
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

conn = mysql.connector.connect(
    host = 'localhost',
    user = 'root',
    password = 'malcolmX@1974',
    database = 'purchases')

cur = conn.cursor()



# List all unique cities where customers are located.

In [ ]:
cur = conn.cursor()

query = """ select distinct City from salless """

df = pd.read_sql(query, conn)

cur.execute(query)

data = cur.fetchall()

data


# Count the number of orders placed in 2015.

In [ ]:
cur = conn.cursor()

query = """ select count(order_id) from sales where year(order_date) = 2015 """

cur.execute(query)

data = cur.fetchall()

"Total number of sales in 2015 are", data[0][0]

# Find the total sales per category

In [ ]:
cur = conn.cursor()

query = """ SELECT upper(`Sub-Category`), round(sum(Sales),3) as total_sales_per_category FROM purchases.saless
group by `Sub-Category`
order by total_sales_per_category desc """

cur.execute(query)

data = cur.fetchall()

df = pd.DataFrame(data, columns = ["Category", "Sales"])

df


 # Count the number of customers from each state. 


In [ ]:
cur = conn.cursor()

query = """ SELECT State, count(State) as total_customers_per_state FROM purchases.saless
group by State
order by total_customers_per_state desc """

cur.execute(query)

data = cur.fetchall()

df = pd.DataFrame(data, columns = ["State", "Population"])

plt.figure(figsize = (9,3))
plt.bar(df["State"], df["Population"])
plt.xticks(rotation = 90)
plt.xlabel("States")
plt.ylabel("Population")
plt.title("Count of Customers per State")
plt.show()




# Calculate the number of orders per month in 2018

In [ ]:
cur = conn.cursor()
query = """ select monthname(order_date) as months, count(order_id) as order_count from purchases.sales
group by months
order by order_count desc """

cur.execute(query)

data = cur.fetchall()
df = pd.DataFrame(data, columns = ["months", "order_count"])

b = ["January", "February", "March", "April", "May", "June","July", "August", "September", "October", "November", "December"]

ax = sns.barplot(x = df["months"], y = df["order_count"], data = df, order = b)

plt.xticks(rotation = 45)
plt.title("Count of customer orders per month in 2016")
ax.bar_label(ax.containers[0])
plt.show()

# Find the average number of products per order, grouped by customer city

In [ ]:
cur = conn.cursor()
query = """ select City, round(avg(total_order), 2) as avg_products_per_order from (
select City, order_id, sum(Quantity) as total_order from purchases.saless
group by City, order_id
) as sub
group by City
order by avg_products_per_order desc;
#SELECT City, round(avg(quantity), 2) as average_orders FROM purchases.saless
#group by City
#order by average_orders desc """

cur.execute(query)

data = cur.fetchall()
df = pd.DataFrame(data, columns = ["City" , "Average Order"])
df.head(11)

# Calculate the percentage of total revenue contributed by each product category

In [ ]:
cur = conn.cursor()

cur.execute(query)

query = """ select Category, round((sum(Sales)/(select sum(Sales) from purchases.saless))* 100, 2) 
as sales_percentage from purchases.saless
group by Category
order by sales_percentage desc """

data = cur.fetchall()


df = pd.DataFrame(data, columns = ["Category", "percentage distribution"])


c = ["blue", "red", "silver"]
ex = [0,0,0,0.1,0]
plt.pie(df["percentage distribution"], labels = df["Category"], autopct = "%.2f", colors = c, shadow = True)
plt.show()

# Identify the correlation between product price and the number of times a product has been purchased.


In [ ]:
cur = conn.cursor()
query = """ select upper(`Sub-Category`),count(product_id), round(avg(Sales * Quantity), 2) from purchases.saless
group by `Sub-Category` """

cur.execute(query)
data = cur.fetchall()
data
df = pd.DataFrame(data, columns = ["Sub-Category", "Product count", "Average Price"])

arr1 = df["Product count"]
arr2 = df["Average Price"]
a = np.corrcoef([arr1, arr2])

print("The correlation between product price and the number of times a product has been purchased is", a[0][1])

# Calculate the total revenue generated by each seller, and rank them by revenue.


In [ ]:
cur = conn.cursor()
query = """ select * , dense_rank() over(order by revenue_generated desc) as rn from 
(SELECT customer_name, round(sum(Sales), 2) as revenue_generated FROM purchases.saless
group by customer_name
order by revenue_generated desc) as a"""

cur.execute(query)
data = cur.fetchall()
df = pd.DataFrame(data, columns = ["customer_name", "revenue", "rank"])
df = df.head(11)
sns.barplot(x = "customer_name", y = "revenue", data = df)
plt.xticks(rotation = 90)
plt.show()

# Calculate the moving average of order values for each customer over their order history.


In [ ]:
cur = conn.cursor()
query = """ select customer_id, order_date, payment,
round(avg(payment) over(partition by customer_id order by order_date
rows between 2 preceding and current row), 2) as mov_avg
from 
(select customer_id, order_date, 
Sales as payment from purchases.sales) as a """

cur.execute(query)
data = cur.fetchall()

df = pd.DataFrame(data, columns = ["customer_id", "order_date", "payment", "moving_average"])
df

# Calculate the cumulative sales per month for each year

In [ ]:
cur = conn.cursor()
query = """ select years, months, payment, round(sum(payment)
over(order by years, months), 2) as culmulative_sales
from
(select year(order_date) as years,
month(order_date) as months,
round(sum(Sales), 2) as payment from purchases.sales
group by years, months order by years, months) as a """

cur.execute(query)
data = cur.fetchall()

df = pd.DataFrame(data, columns = ["years", "months", "payment", "culmulative sales"])
df

# Calculate the year-over-year growth rate of total sales

In [ ]:
cur = conn.cursor()
query = """with a as(select year(order_date) as years,
round(sum(`Sales`), 2) as payment from purchases.sales
group by years order by years) 

select years, ((payment - lag(payment, 1) over(order by years))/
lag(payment, 1) over(order by years)) * 100 from a"""

cur.execute(query)
data = cur.fetchall()
df = pd.DataFrame(data, columns = ["years", "% growth rate"])
df

# Identify the top 3 customers who spent the most money in each year.

In [ ]:
cur = conn.cursor()
query = """ select years, customer_name, payment, d_rank
from
(select year(order_date) as years, customer_name, round(sum(Sales), 2) as payment,
dense_rank() over(partition by year(order_date) order by sum(Sales) desc) d_rank from purchases.sales
group by year(order_date), customer_name) as a
where d_rank <= 3 """

cur.execute(query)
data = cur.fetchall()
df = pd.DataFrame(data, columns = ["years", "customer_name", "revenue", "rank"])
sns.barplot(x = "customer_name", y = "revenue", data = df, hue = "years")
plt.xticks(rotation =90)
plt.show()